# Friday Night Frisbee Video Browser

This notebook builds a standalone HTML page with a YouTube player beside a Shown Space-style field. Clicking a throw seeks the video to an estimated timestamp from manually entered point-start anchors.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, "../src")

from ufa import (
    add_estimated_video_seconds,
    build_fnf_browser_data,
    build_fnf_game_table,
    load_fnf_point_anchors,
    write_fnf_video_browser_html,
)

## Find Friday Night Frisbee Games

This searches the UFA schedule metadata for Friday games, then joins in manually verified Friday Night Frisbee YouTube URLs from `../data/manual/fnf_games.csv`. The UFA metadata is useful for finding Friday games, but the broadcast URL often points to WatchUFA rather than the free YouTube video.

In [4]:
FNF_START_DATE = "2026-04-24"
FNF_END_DATE = "2026-08-01"
TEAM_ID = None  # Leave as None for both teams, or set to a team id like "glory".
FNF_GAMES_CSV = Path("../data/manual/fnf_games.csv")

fnf_games = build_fnf_game_table(
    FNF_START_DATE,
    FNF_END_DATE,
    schedule_path=FNF_GAMES_CSV,
    season=2026,
    team_id=TEAM_ID,
)

fnf_games

,gameID,awayTeamID,homeTeamID,startTimestamp,week,status,location,youtube_url,is_fnf_youtube,fnf_source,streamingURL
0,2026-04-24-CAR-SD,flyers,growlers,2026-04-24T19:00:00-07:00,week-1,Final,Mission Bay High School,https://www.youtube.com/watch?v=TY7WdnObrKw,True,YouTube oEmbed,https://watchufa.tv/videos/carolina-at-san-die...
1,2026-04-24-ATL-HTX,hustle,havoc,2026-04-24T19:30:00-05:00,week-1,Final,Mercer Stadium,,False,,https://watchufa.tv/videos/atlanta-at-houston-...
2,2026-05-01-TOR-NY,rush,empire,2026-05-01T19:00:00-04:00,week-2,Final,The Stadium at Memorial Field,,False,,https://watchufa.tv/videos/toronto-at-new-york...
3,2026-05-01-ORE-SLC,steel,shred,2026-05-01T19:00:00-06:00,week-2,Final,Zion Bank Stadium,,False,,https://watchufa.tv/videos/oregon-at-salt-lake...
4,2026-05-08-ATL-LV,hustle,bighorns,2026-05-08T19:00:00-07:00,week-3,Final,Bonanza High School,,False,,https://watchufa.tv/videos/atlanta-at-las-vega...
5,2026-05-15-COL-DC,apex,breeze,2026-05-15T19:00:00-04:00,week-4,Final,"Carlini Field, Catholic U",,False,,https://watchufa.tv/videos/colorado-at-dc-5-15...
6,2026-05-15-MAD-PIT,radicals,thunderbirds,2026-05-15T19:00:00-04:00,week-4,Final,F.N.B. Stadium,,False,,https://watchufa.tv/videos/madison-at-pittsbur...
7,2026-05-15-MIN-IND,windchill,alleycats,2026-05-15T19:00:00-04:00,week-4,Final,Kuntz Stadium,,False,,https://watchufa.tv/videos/minnesota-at-indian...
8,2026-05-15-LV-HTX,bighorns,havoc,2026-05-15T19:30:00-05:00,week-4,Final,Dulles High School,,False,,https://watchufa.tv/videos/las-vegas-at-housto...
9,2026-05-29-DC-TOR,breeze,rush,2026-05-29T19:00:00-04:00,week-6,Final,Varsity Stadium,,False,,https://watchufa.tv/videos/dc-at-toronto-5-29-...


## Choose A Friday Night Frisbee Game

Pick the row number from `fnf_games` that you want to turn into a browser. Rows where `is_fnf_youtube` is `True` have a usable YouTube URL. If the row you want is `False`, add that game and YouTube URL to `../data/manual/fnf_games.csv`, then rerun the table cell.

In [11]:
GAME_ROW = 0

if fnf_games.empty:
    raise ValueError("No Friday games were found for this date range.")

selected_game = fnf_games.iloc[GAME_ROW]
GAME_ID = selected_game["gameID"]
YOUTUBE_URL = selected_game["youtube_url"]

if not isinstance(YOUTUBE_URL, str) or not YOUTUBE_URL.strip():
    raise ValueError("This row does not have a YouTube URL yet. Add it to ../data/manual/fnf_games.csv, then rerun the FNF table cell.")

ANCHORS_CSV = Path("../data/manual/fnf_point_anchors.csv")
OUTPUT_HTML = Path(f"../outputs/fnf_browsers/{GAME_ID}.html")

print(GAME_ID)
print(YOUTUBE_URL)
selected_game

2026-04-24-CAR-SD
https://www.youtube.com/watch?v=TY7WdnObrKw


gameID                                            2026-04-24-CAR-SD
awayTeamID                                                   flyers
homeTeamID                                                 growlers
startTimestamp                            2026-04-24T19:00:00-07:00
week                                                         week-1
status                                                        Final
location                                    Mission Bay High School
youtube_url             https://www.youtube.com/watch?v=TY7WdnObrKw
is_fnf_youtube                                                 True
fnf_source                                           YouTube oEmbed
streamingURL      https://watchufa.tv/videos/carolina-at-san-die...
Name: 0, dtype: object

## Manual Point Anchors

Add point-start anchors to `../data/manual/fnf_point_anchors.csv`. Each row should say where a game point starts in the YouTube video.

Required columns:

`game_id, youtube_url, game_quarter, quarter_point, video_seconds, note`

In [12]:
anchors = load_fnf_point_anchors(ANCHORS_CSV, GAME_ID)
anchors

,game_id,youtube_url,game_quarter,quarter_point,video_seconds,note


## Fetch Throws And Build Browser Data

In [13]:
possessions, paths = build_fnf_browser_data(GAME_ID, team_id=TEAM_ID)
paths = add_estimated_video_seconds(paths, anchors, seconds_per_throw=3.0)

print(f"Scoring possessions: {len(possessions):,}")
print(f"Paths: {len(paths):,}")
possessions.head()

Scoring possessions: 45
Paths: 45


,possession_id,GameID,team_id,start_timestamp,game_quarter,quarter_point,possession_num,is_home_team,start_x,start_y,...,mean_cp,risk_adjusted_aec_per_throw,total_yards,yards_per_throw,total_throw_distance,avg_throw_distance,max_throw_distance,huck_count,reset_count,lateral_yards
0,2026-04-24-CAR-SD|1|1|1|False,2026-04-24-CAR-SD,flyers,2026-04-24 19:00:00,1,1,1,False,15.61,92.18,...,0.954503,0.021951,13.36,1.027692,249.821295,19.217023,44.842788,1,8,135.94
1,2026-04-24-CAR-SD|1|2|1|True,2026-04-24-CAR-SD,growlers,2026-04-24 19:00:00,1,2,1,True,17.18,25.29,...,0.961522,0.082591,78.82,6.568333,160.932992,13.411083,42.559588,1,3,110.28
2,2026-04-24-CAR-SD|1|3|1|False,2026-04-24-CAR-SD,flyers,2026-04-24 19:00:00,1,3,1,False,-0.82,30.61,...,0.972228,0.060823,75.20,5.013333,204.348668,13.623245,26.353387,0,5,127.41
3,2026-04-24-CAR-SD|1|4|1|True,2026-04-24-CAR-SD,growlers,2026-04-24 19:00:00,1,4,1,True,10.36,16.22,...,0.932897,0.154613,84.96,14.160000,112.018500,18.669750,47.033865,1,0,60.14
4,2026-04-24-CAR-SD|1|5|2|True,2026-04-24-CAR-SD,growlers,2026-04-24 19:00:00,1,5,2,True,11.73,58.36,...,0.963826,0.120557,48.34,6.042500,100.559750,12.569969,27.618373,0,3,53.73


## Export HTML Browser

In [14]:
output_path = write_fnf_video_browser_html(
    game_id=GAME_ID,
    youtube_url=YOUTUBE_URL,
    possessions=possessions,
    paths=paths,
    output_path=OUTPUT_HTML,
)

local_url = "http://localhost:8000/" + output_path.resolve().relative_to(Path.cwd().parent).as_posix()
print("Start this server from the project root in a terminal:")
print("python -m http.server 8000")
print("Then open:")
print(local_url)

output_path

WindowsPath('../outputs/fnf_browsers/2026-04-24-CAR-SD.html')

Do not open the generated HTML as a local `file://` page. YouTube may show `Error 153` when the page has no normal web origin.

Instead, from the project root run:

```bash
python -m http.server 8000
```

Then open the printed `http://localhost:8000/...` URL. In the browser, choose a possession, click a throw, and the YouTube player will seek to that estimated moment. Left/Right arrow keys move through throws after focusing the field.